# 阿里云算法岗真题

解析参考：[塔子哥学算法](https://mp.weixin.qq.com/s/lvnLU1bOG29DF7QPC3fQpA)

## 2025-04-24-电子商务平台

### 题目描述

某电子商务平台希望使用机器学习来改善用户的购物体验。他们收集了用户的购买历史数据，希望你能构建一个推荐系统，为用户推荐他们可能喜欢的商品。

你的任务是，使用$scikit-learn$库，基于用户的购买历史数据，构建一个基于$K$最近邻（$KNN$）的推荐系统，并使用$Cosine$相似度来评估模型的性能。

要求：

1. 首先要计算用户之间的$Cosine$相似度矩阵，然后在推荐系统中使用基于$KNN$的方法，并且将$KNN$的距离度量设置为$cosine$。

2. $KNN$的邻居数量${n_{neighbors}}$参数设为4，其中包括用户自己，最后在生成推荐时排除自己（即实际使用3个邻居）。

3. 推荐的商品必须是用户尚未购买过的商品（即排除已购买的商品）。

4. 如果没有合适的商品推荐（例如用户的邻居没有新商品），输出结果中应显示$None$和相似度为$0.0$。

5. 相同相似度情况下，取$ID$最小的商品。

### 输入描述

输入是一个$list$，每个元素是一个包含两个元素的$list$，第一个元素是用户$ID$，第二个元素是该用户购买过的商品$ID$的列表。

### 输出描述

输出是一个$list$，包含三个元素，第一个元素是用户$ID$，第二个元素是预测的用户可能喜欢的商品$ID$，第三个元素是预测的$Cosine$相似度，保留小数点后3位有效数字。

### 补充说明

(1) 可以使用形如$numpy$、$pandas$、$sklearn$的第三方代码库。

(2) 为了保证唯一性，请严格遵循输入输出描述进行作答。

(3) 形如$sys.stdin$等方法结合$for$循环和$eval$函数即可读取并还原输入数据。

(4) 算法使用$brute$，最近邻数量设置为3。


### 样例

```
输入:
[[1,[101,102,103]],[2,[101,104]],[3,[102,103,105]],[4,[101,103,105]]]

输出:
[1,105,0.667]
[2,102,0.408]
[3,101,0.667]
[4,102,0.667]

```

In [2]:
import numpy as np
from sklearn.neighbors import NearestNeighbors

def main():
    input_raw = eval(input()) # [[1,[101,102,103]],[2,[101,104]],[3,[102,103,105]],[4,[101,103,105]]]
    uids = [u[0] for u in input_raw]
    items = [u[1] for u in input_raw]
    items_sort = sorted({item for temp in items for item in temp}) # 集合
    items_idx = {v : i for i, v in enumerate(items_sort)} # 创建索引

    X = np.zeros((len(uids), len(items_sort)), dtype=int) # 构造用户-商品二值矩阵

    for i, its in enumerate(items):
        for it in its:
            X[i, items_idx[it]] = 1

    knn = NearestNeighbors(n_neighbors=4, metric='cosine', algorithm='brute')
    knn.fit(X)
    dists, idxs = knn.kneighbors(X) # 返回邻居列表, 第一个是自己

    for i, its in enumerate(items): # 开始推荐
        uid = uids[i]
        neighbors = idxs[i][1:]
        similaritys = 1 - dists[i][1:]
        seen = set(its)
        candidates = {}
        for nb, sim in zip(neighbors, similaritys):
            for item in items[nb]: # 寻找邻居中未见过的商品
                if item not in seen: 
                    if sim > candidates.get(item, 0):
                        candidates[item] = sim # 记录未见过商品的相似度
        if candidates:
            max_sim = max(candidates.values()) # 找到最大相似度
            rec_id = min([idx for idx, sim in candidates.items() if sim == max_sim]) # 最大相似度中的最小id
        else:
            rec_id = None
            max_sim = 0.0
        print(f"[{uid},{rec_id},{max_sim:.3f}]")
    
if __name__ == "__main__":
    # 数据输入
    main()
    

In [3]:
input_raw = eval(input()) # [[1,[101,102,103]],[2,[101,104]],[3,[102,103,105]],[4,[101,103,105]]]
uids = [u[0] for u in input_raw]
items = [u[1] for u in input_raw]

In [4]:
print(items)

[[101, 102, 103], [101, 104], [102, 103, 105], [101, 103, 105]]


In [5]:
items_sort = sorted({item for temp in items for item in temp}) # 集合
print(items_sort)
items_idx = {v : i for i, v in enumerate(items_sort)} # 创建索引
print(items_idx)

[101, 102, 103, 104, 105]
{101: 0, 102: 1, 103: 2, 104: 3, 105: 4}


In [6]:
X = np.zeros((len(uids), len(items_sort)), dtype=int) # 构造用户-商品二值矩阵

for i, its in enumerate(items):
    for it in its:
        X[i, items_idx[it]] = 1
X

array([[1, 1, 1, 0, 0],
       [1, 0, 0, 1, 0],
       [0, 1, 1, 0, 1],
       [1, 0, 1, 0, 1]])

In [7]:
knn = NearestNeighbors(n_neighbors=4, metric='cosine', algorithm='brute')
knn.fit(X)
dists, idxs = knn.kneighbors(X) # 返回邻居列表, 第一个是自己
print(dists)
print(idxs)

[[0.         0.33333333 0.33333333 0.59175171]
 [0.         0.59175171 0.59175171 1.        ]
 [0.         0.33333333 0.33333333 1.        ]
 [0.         0.33333333 0.33333333 0.59175171]]
[[0 3 2 1]
 [1 0 3 2]
 [2 0 3 1]
 [3 0 2 1]]


In [11]:
for i, its in enumerate(items):
    uid = uids[i]
    neighbors = idxs[i][1:]
    similaritys = 1 - dists[i][1:]
    seen = set(its)
    candidates = {}
    for nb, sim in zip(neighbors, similaritys):
        for item in items[nb]: # 寻找邻居中未见过的商品
            if item not in seen: 
                if sim > candidates.get(item, 0):
                    candidates[item] = sim # 记录未见过商品的相似度
    if candidates:
        max_sim = max(candidates.values()) # 找到最大相似度
        rec_id = min([idx for idx, sim in candidates.items() if sim == max_sim]) # 最大相似度中的最小id
    else:
        rec_id = None
        max_sim = 0.0
    print(f"[{uid},{rec_id},{max_sim:.3f}]")

[1,105,0.667]
[2,102,0.408]
[3,101,0.667]
[4,102,0.667]


## 题目解析

### 构建用户-商品矩阵

首先将输入的用户购买历史转换为用户-商品的二值矩阵。行表示用户，列表示商品，如果用户购买过该商品则对应位置为1，否则为0。

### 计算$Cosine$相似度与$KNN$邻居

使用$scikit-learn$的$NearestNeighbors$，设置：
- $metric='cosine'$ （返回的距离为 $1-cosine\_similarity$）
- $algorithm='brute'$
- $n\_neighbors=4$ （包括自身在内共4个邻居）

对每个用户调用$kneighbors$得到最近4个邻居，排除自身后实际使用3个邻居，并将距离转换为相似度：

$$
sim = 1.0 - distance
$$

### 生成推荐

对于每个用户：

1. 遍历3个邻居，将邻居购买但当前用户未购买的商品作为候选
2. 对每个候选商品，记录其与当前用户之间的最大相似度
3. 从候选集中选出相似度最高的商品；若存在多件，则取商品$ID$最小者
4. 若无任何候选商品，则输出 $None$，相似度设为 $0.000$

### 复杂度分析

- 构建矩阵耗时 $O(NM)$，其中 $N$ 为用户数，$M$ 为商品数。
- $KNN$ 基于 $brute force$，计算所有用户间距离，时间复杂度 $O(N^2M)$。
- 最后生成推荐阶段，遍历每个用户的3个邻居并筛选商品，时间 $O(NKQ)$（$K=3$，$Q$ 为平均邻居商品数）。
- 整体时间复杂度以 $O(N^2M)$ 为主；空间复杂度为 $O(NM)$。

Brute Force方法的核心思想是通过计算目标点与数据集中每个点之间的距离，然后选择距离最近的K个点。具体步骤如下：
1. 计算距离：对于数据集中的每个点，计算其与目标点之间的距离。常用的距离度量包括欧氏距离、曼哈顿距离、余弦距离等。
2. 选择最近的K个点：将计算得到的距离进行排序，选择距离最小的K个点。
3. 预测结果：对于分类任务，根据这K个最近邻点的标签进行多数投票；对于回归任务，则取这K个最近邻点的值的平均值。